<a href="https://colab.research.google.com/github/Spandana-mr/6thSem-ML-Lab/blob/main/Lab_10_ML_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# ================================
# 2. Load Dataset
# ================================
df = pd.read_csv("heart.csv")

print("Dataset Preview:\n", df.head())
print("\nColumns:\n", df.columns)

# ================================
# 3. Separate Features and Target
# ================================
X = df.drop('HeartDisease', axis=1)   # change 'target' if column name differs
y = df['HeartDisease']

# ================================
# 4. Handle Categorical Data
# ================================
# Identify categorical columns
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# One-Hot Encoding for categorical columns
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

X = ct.fit_transform(X)

# ================================
# 5. Feature Scaling
# ================================
scaler = StandardScaler()
X = scaler.fit_transform(X)

# ================================
# 6. Train-Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 7. Train Models (Before PCA)
# ================================
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier()
}

print("\n--- Accuracy BEFORE PCA ---")
before_pca_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    before_pca_results[name] = acc
    print(f"{name}: {acc:.4f}")

# ================================
# 8. Apply PCA
# ================================
pca = PCA(n_components=0.95)  # retain 95% variance
X_pca = pca.fit_transform(X)

print("\nOriginal Shape:", X.shape)
print("Reduced Shape:", X_pca.shape)

# ================================
# 9. Train-Test Split (PCA Data)
# ================================
X_train_pca, X_test_pca, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42
)

# ================================
# 10. Train Models (After PCA)
# ================================
print("\n--- Accuracy AFTER PCA ---")
after_pca_results = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    after_pca_results[name] = acc
    print(f"{name}: {acc:.4f}")

# ================================
# 11. Comparison Table
# ================================
print("\n--- Final Comparison ---")
for model in models.keys():
    print(f"{model}: Before PCA = {before_pca_results[model]:.4f}, After PCA = {after_pca_results[model]:.4f}")

Dataset Preview:
    Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  

Columns:
 Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
    